# NaturalisticDiffInt — 03: Brain Comparison

**Goal:** Compare NMPH model RSA predictions with StudyForrest fMRI data (single run, exploratory).

1. Load model results from notebook 02 (model RSA change + brain RSA, GSBS event structure)
2. Reload GSBS brain data; compute per-parcel event-level RSA
3. Parcel-level model–brain RSA correlation (Spearman r, Schaefer 400)
4. Network-level summary (Yeo 17 networks)
5. **H4:** Global brain state modulation — derive TPN/DMN score from GSBS state patterns
6. **H1:** Hippocampal RSA for model-differentiated vs model-integrated pairmate pairs

**Single-run note:** With one run we cannot compute cross-run RSA *change*.
Instead, H1 tests whether model-differentiated pairs already show *lower* hippocampal RSA
within this run — a within-run proxy for the expected longitudinal pattern.


## 0. GitHub Sync — Setup

In [ ]:
from google.colab import userdata
import os, subprocess

GITHUB_USER  = "drgzkr"
GITHUB_REPO  = "NaturalisticDiffInt"
REPO_PATH    = f"/content/{GITHUB_REPO}"
NOTEBOOK_REL = "notebooks/analysis/03_brain_comparison.ipynb"

_token  = userdata.get("GITHUB_TOKEN")
_remote = f"https://{_token}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

subprocess.run(["git", "config", "--global", "user.name", "Colab"], check=True)
subprocess.run(["git", "config", "--global", "user.email", "colab@naturalistic-diffint.local"], check=True)

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", _remote, REPO_PATH], check=True)
    print(f"Cloned  -> {REPO_PATH}")
else:
    subprocess.run(["git", "-C", REPO_PATH, "remote", "set-url", "origin", _remote])
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
    print(f"Pulled  -> {REPO_PATH}")

del _token, _remote
print(f"Repo ready at {REPO_PATH}")


## 1. Mount Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT   = "/content/project_data"           # GSBS objects (downloaded in section 4 below)
RESULTS_DIR = "/content/drive/MyDrive/NaturalisticDiffInt/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Match the SUB/RUN/MODALITY used in notebook 02
SUB              = 'sub-01'
RUN              = 1
FEATURE_MODALITY = 'qwen'    # must match notebook 02
N_ROIS           = 400
TR               = 2.0

res_path = f"{RESULTS_DIR}/nmph_naturalistic_{FEATURE_MODALITY}_{SUB}_run{RUN}.pkl"
print(f"Model results file: {res_path}")
print(f"Exists: {os.path.exists(res_path)}")
if not os.path.exists(res_path):
    print("\nRun notebook 02 first to generate this file.")


## 2. Imports

In [ ]:
import sys
sys.path.insert(0, REPO_PATH)

import pickle, warnings
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
np.random.seed(42)

print("Imports OK.")


## 3. Load model results from notebook 02

In [ ]:
with open(res_path, 'rb') as f:
    pkg = pickle.load(f)

rsa_before      = pkg['rsa_before']       # (n_events, n_events)
rsa_after       = pkg['rsa_after']
rsa_change      = pkg['rsa_change']       # model RSA change matrix
brain_rsa       = pkg['brain_rsa']        # full-brain cosine RSA from GSBS BOLD
model_log       = pkg['log']              # list of episode dicts
event_onsets    = pkg['event_onsets']     # (n_events,)
event_durations = pkg['event_durations']  # (n_events,)
gsbs_states     = pkg['gsbs_states']      # (n_trs,) state label per TR
gsbs_deltas     = pkg['gsbs_deltas']      # (n_trs,) boundary strength
cfg             = pkg['config']
n_events        = pkg['n_events']

print("Model config:", cfg)
print(f"Events        : {n_events}")
print(f"RSA change    : {rsa_change.shape}")
print(f"Brain RSA     : {brain_rsa.shape}")
print(f"Log episodes  : {len(model_log)}")


In [ ]:
# Auto-print: model configuration recap
all_dirs   = [e['direction'] for e in model_log]
all_deltas = [e['delta_sim'] for e in model_log]
n_diff  = all_dirs.count('differentiation')
n_intg  = all_dirs.count('integration')
n_nc    = all_dirs.count('no_change')
n_total = len(model_log)

print("=" * 60)
print(f"MODEL RESULTS RECAP  ({SUB}  run {RUN}  {FEATURE_MODALITY.upper()})")
print("=" * 60)
print(f"  N events              : {n_events}")
print(f"  N competition episodes: {n_total}")
print(f"  Differentiation       : {n_diff} ({n_diff/max(n_total,1)*100:.1f}%)")
print(f"  Integration           : {n_intg} ({n_intg/max(n_total,1)*100:.1f}%)")
print(f"  No change             : {n_nc}  ({n_nc/max(n_total,1)*100:.1f}%)")
print(f"  Mean delta_sim        : {(sum(all_deltas)/len(all_deltas)) if all_deltas else 0:+.4f}")
print(f"  Competitor threshold  : {cfg['competitor_threshold']}")
print(f"  N_CATEGORY            : {cfg['n_category']}")
print(f"  osc_amp               : {cfg['osc_amp']}")
print(f"  N_ROIS                : {cfg['n_rois']}")
print("=" * 60)

if n_total == 0:
    print("\nWARNING: No competition episodes. Re-run notebook 02 with a lower COMPETITOR_THRESH.")


**Before continuing:** Confirm the configuration above matches notebook 02. If `N episodes = 0`, go back to notebook 02 and lower `COMPETITOR_THRESH` (try 0.3–0.4 for Qwen).

In [ ]:
!pip install -q git+https://github.com/drgzkr/statesegmentation.git


## 4. Download GSBS brain data and compute per-parcel event RSA

The GSBS objects are downloaded here independently — each Colab session has its own filesystem and cannot share files with notebook 02.

Each `.npy` file contains a fitted GSBS object for one subject × run, with:
- **`.x`** — ROI × time BOLD timeseries (`n_rois × n_trs`), z-scored Schaefer parcels
- **`.states`** — state label per TR
- **`.state_patterns`** — mean BOLD pattern per state (`n_states × n_rois`)

The model results (pkl) are loaded from Google Drive above — only the raw BOLD is re-fetched here for parcel-level RSA computation.


In [ ]:
import zipfile

GSBS_ZIP_ID   = "10Mms7DyQ87mq-jS8W7KfscGmc5QZhdmk"
GSBS_ZIP_PATH = os.path.join(DATA_ROOT, "StudyForrest_SingleSubGlobalGSBS_Results.zip")
GSBS_DIR      = os.path.join(DATA_ROOT, "StudyForrest_SingleSubGlobalGSBS_Results")
os.makedirs(DATA_ROOT, exist_ok=True)

if not os.path.exists(GSBS_DIR):
    if not os.path.exists(GSBS_ZIP_PATH):
        print("Downloading GSBS objects (~167 MB)...")
        import subprocess
        subprocess.run(["pip", "install", "-q", "gdown"], check=True)
        import gdown
        gdown.download(id=GSBS_ZIP_ID, output=GSBS_ZIP_PATH, quiet=False)
    print("Unzipping...")
    with zipfile.ZipFile(GSBS_ZIP_PATH, "r") as zf:
        zf.extractall(DATA_ROOT)
    print(f"Done -> {GSBS_DIR}")
else:
    print(f"GSBS data already present at {GSBS_DIR}")


In [ ]:
GSBS_DIR = os.path.join(DATA_ROOT, "StudyForrest_SingleSubGlobalGSBS_Results")

def load_gsbs(sub, run, n_rois=400):
    path = os.path.join(GSBS_DIR, f"GSBS_{sub}_run{run}Schaefer_{n_rois}_ROIs.npy")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"GSBS file not found: {path}\nRun notebook 02 first to download GSBS data.")
    return np.load(path, allow_pickle=True).item()

gsbs = load_gsbs(SUB, RUN, N_ROIS)
bold = gsbs.x            # (n_rois, n_trs)
state_patterns = gsbs.state_patterns  # (n_states, n_rois)
print(f"BOLD shape     : {bold.shape}  (n_rois x n_trs)")
print(f"State patterns : {state_patterns.shape}")

HRF_DELAY = 2  # TRs

def event_avg_bold(bold_roi_time, onsets, durations, hrf_delay=HRF_DELAY):
    """bold_roi_time: (n_rois, n_trs). Returns (n_events, n_rois)."""
    n_trs = bold_roi_time.shape[1]
    ev = np.zeros((len(onsets), bold_roi_time.shape[0]), dtype=np.float32)
    for i, (s, d) in enumerate(zip(onsets, durations)):
        start = min(int(s) + hrf_delay, n_trs - 1)
        end   = min(int(s + d) + hrf_delay, n_trs)
        ev[i] = bold_roi_time[:, start:end].mean(1)
    return ev

ev_bold = event_avg_bold(bold, event_onsets, event_durations)
print(f"Event BOLD     : {ev_bold.shape}  (n_events x n_rois)")


## 5. Parcel-level model–brain RSA correlation

For each parcel: Pearson RSA between events (outer product of z-scored activation), then
Spearman r against the model RSA change upper triangle. FDR correction (Benjamini-Hochberg).


In [ ]:
def upper_tri(mat):
    n = mat.shape[0]
    idx = np.triu_indices(n, k=1)
    return mat[idx]

model_change_vec = upper_tri(rsa_change)

print(f"Computing parcel-level RSA correlations ({N_ROIS} parcels)...")

r_map = np.zeros(N_ROIS)
p_map = np.zeros(N_ROIS)

for p in range(N_ROIS):
    col = ev_bold[:, p]
    if col.std() < 1e-8:
        r_map[p] = np.nan; p_map[p] = np.nan; continue
    c = (col - col.mean()) / col.std()
    parcel_rsa = np.outer(c, c)
    brain_vec  = upper_tri(parcel_rsa)
    if brain_vec.std() < 1e-8:
        r_map[p] = np.nan; p_map[p] = np.nan; continue
    r, pv = spearmanr(model_change_vec, brain_vec)
    r_map[p] = r; p_map[p] = pv

valid = ~np.isnan(p_map)
_, p_fdr, _, _ = multipletests(p_map[valid], method='fdr_bh')
p_map_fdr = np.full(N_ROIS, np.nan)
p_map_fdr[valid] = p_fdr

sig_parcels = np.where((p_map_fdr < 0.05) & valid)[0]
print(f"Significant parcels (FDR q<0.05): {len(sig_parcels)} / {N_ROIS}")
print(f"Max r = {np.nanmax(r_map):.3f}  |  Min r = {np.nanmin(r_map):.3f}  |  Mean r = {np.nanmean(r_map):.3f}")


In [ ]:
# Auto-print: parcel RSA correlation summary
valid_mask = ~np.isnan(r_map)
r_valid = r_map[valid_mask]

print("=" * 65)
print("PARCEL-LEVEL MODEL-BRAIN RSA CORRELATION")
print(f"  ({SUB}  run {RUN}  {FEATURE_MODALITY.upper()})")
print("=" * 65)
print(f"  Parcels computed         : {valid_mask.sum()} / {len(r_map)}")
print(f"  Significant (FDR q<0.05): {len(sig_parcels)}")
print(f"  Mean r (all parcels)     : {r_valid.mean():+.4f}  (SD={r_valid.std():.4f})")
print(f"  Max r                    : {r_valid.max():+.4f}  (parcel {np.nanargmax(r_map)})")
print(f"  Min r                    : {r_valid.min():+.4f}  (parcel {np.nanargmin(r_map)})")
print(f"  Parcels with r > 0       : {(r_valid>0).sum()} ({(r_valid>0).mean()*100:.1f}%)")
print(f"  Parcels with r < 0       : {(r_valid<0).sum()} ({(r_valid<0).mean()*100:.1f}%)")
print()

if n_total == 0:
    print("WARNING: No competition episodes -- parcel map is uninformative.")
elif len(sig_parcels) == 0:
    print("  No significant parcels after FDR correction.")
    print("  With a single run, power is limited. Inspect the raw r distribution below.")
    print("  Consider: lower COMPETITOR_THRESH in notebook 02, or try VGG-19 features.")
elif len(sig_parcels) < 10:
    print(f"  {len(sig_parcels)} significant parcels -- sparse signal present.")
    print("  Check whether they cluster in hippocampus/DMN (H1) or TPN (H2).")
else:
    print(f"  {len(sig_parcels)} significant parcels -- meaningful spatial structure.")

dom = 'differentiation' if n_diff > n_intg else 'integration'
mean_r = r_valid.mean()
if dom == 'differentiation' and mean_r < 0:
    print("\n  Consistent with H1: differentiation-dominant model; negative mean r.")
elif dom == 'integration' and mean_r > 0:
    print("\n  Consistent with H2: integration-dominant model; positive mean r.")
else:
    print(f"\n  Mean r {mean_r:+.4f} with dominant outcome '{dom}'.")
    print("  Regional variation likely -- inspect the network plot in next section.")
print("=" * 65)


**How to read the parcel r-map:** Positive r = regions where model-integrated pairs show high brain-side similarity (semantic generalization areas). Negative r = regions where model-differentiated pairs show low brain-side similarity — the expected H1 signature in hippocampus-adjacent areas. Near-zero r = regions not sensitive to NMPH dynamics for this feature space.

## 6. Parcel r-map visualisation

In [ ]:
sorted_idx = np.argsort(r_map)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
top_n = 50
colors_top = ['firebrick' if r > 0 else 'steelblue' for r in r_map[sorted_idx[:top_n]]]
ax.bar(range(top_n), r_map[sorted_idx[:top_n]], color=colors_top, alpha=0.8)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel(f"Parcel rank (top {top_n})")
ax.set_ylabel("Spearman r (model RSA change ~ brain RSA)")
ax.set_title(f"Top {top_n} parcels: model-brain RSA correlation")
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax = axes[1]
ax.hist(r_map[valid_mask], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='k', lw=0.8, ls='--')
mn = np.nanmean(r_map)
ax.axvline(mn, color='red', lw=1.5, label=f"Mean r = {mn:.3f}")
ax.set_xlabel("Spearman r")
ax.set_ylabel("N parcels")
ax.set_title("Distribution of model-brain RSA correlations")
ax.legend()
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle(f"{FEATURE_MODALITY.upper()} | {SUB} run {RUN}", fontsize=11)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/model_brain_r_map_{FEATURE_MODALITY}_{SUB}_run{RUN}.png", dpi=150, bbox_inches='tight')
plt.show()


## 7. Network-level analysis (Yeo 17 networks)

Average r-values within Yeo 17 networks using Schaefer 400 parcel labels from nilearn.
Key predictions: Default/Para (hippocampus-adjacent) networks should show negative r
(H1); Frontoparietal/DAN networks may show positive r (H2/H4 link).


In [ ]:
!pip install nilearn -q

from nilearn.datasets import fetch_atlas_schaefer_2018
atlas = fetch_atlas_schaefer_2018(n_rois=N_ROIS, yeo_networks=17, resolution_mm=2)
labels = [l.decode() if isinstance(l, bytes) else l for l in atlas.labels]

def extract_network(label):
    parts = label.split('_')
    if len(parts) >= 4:
        return '_'.join(parts[3:-1])
    return label

network_labels = [extract_network(l) for l in labels]
unique_networks = sorted(set(network_labels))
print(f"Found {len(unique_networks)} unique Yeo 17 networks")

network_r = {}
for net in unique_networks:
    parcel_mask = np.array([n == net for n in network_labels])
    network_r[net] = np.nanmean(r_map[parcel_mask])

sorted_nets = sorted(network_r.items(), key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(14, 5))
net_names, net_rs = zip(*sorted_nets)
colors = ['firebrick' if r > 0 else 'steelblue' for r in net_rs]
ax.bar(range(len(net_names)), net_rs, color=colors, alpha=0.8)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xticks(range(len(net_names)))
ax.set_xticklabels(net_names, rotation=45, ha='right', fontsize=9)
ax.set_ylabel("Mean Spearman r")
ax.set_title(f"Model-brain RSA correlation by Yeo 17 network  |  {FEATURE_MODALITY.upper()} {SUB} run {RUN}")
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/network_r_map_{FEATURE_MODALITY}_{SUB}_run{RUN}.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Auto-print: network ranking
print("=" * 65)
print("NETWORK-LEVEL RSA CORRELATION RANKING")
print("=" * 65)
print(f"  {'Network':40s}  {'Mean r':>8}  {'Direction':>15}")
print("  " + "-"*63)
for name, r in sorted_nets:
    direction = ("up integration" if r > 0.05 else
                 "down differentiation" if r < -0.05 else "~ no preference")
    marker = "+" if any(k.lower() in name.lower()
                        for k in ['para','default','limbic','hipp']) else " "
    print(f"  {marker} {name:38s}  {r:>+8.4f}  {direction:>20}")
print()
print("  + = hippocampus-adjacent or default-mode network")

top3    = sorted_nets[:3]
bottom3 = sorted_nets[-3:]
print(f"\n  Top 3 (integration-correlated) : {chr(44)+chr(32).join(n for n,_ in top3)}")
print(f"  Bottom 3 (diff-correlated)     : {chr(44)+chr(32).join(n for n,_ in bottom3)}")

hipp_nets = [(n,r) for n,r in sorted_nets
             if any(k.lower() in n.lower() for k in ['para','default','limbic'])]
if hipp_nets:
    hipp_r_mean = sum(r for _,r in hipp_nets) / len(hipp_nets)
    print(f"\n  Hippocampus-adjacent / DMN mean r = {hipp_r_mean:+.4f}")
    if hipp_r_mean < -0.02:
        print("  -> Consistent with H1: DMN/para networks negatively correlated with model change.")
    elif hipp_r_mean > 0.02:
        print("  -> Consistent with H2: DMN networks positively correlated (integration-dominant).")
    else:
        print("  -> No clear DMN preference -- may need more events or different modality.")
print("=" * 65)


**Interpreting the network ranking:** A gradient from DMN/para networks (negative r, H1) to frontoparietal/DAN networks (positive r, H2/H4) is the key spatial prediction. Single-run results will be noisy — directional trends across the network hierarchy are more informative than individual significant parcels at this stage.

## 8. H4: Global brain state modulation

**H4 predicts:** Events encoded during TPN-dominant states (higher attentional arousal) undergo
more differentiation; DMN-dominant states support more integration.

**Approach:** No external global-state file needed. We derive a TPN/DMN balance score for each
GSBS state from `state_patterns` (mean BOLD per state) projected onto the TPN-DMN network axis
defined by Yeo 17 parcel labels. Each event inherits the TPN score of its GSBS state.


In [ ]:
TPN_KEYWORDS = ['DorsAttn', 'SalVentAttn', 'Cont', 'FrontPar']
DMN_KEYWORDS = ['Default', 'Para', 'Limbic']

tpn_mask = np.array([any(k.lower() in l.lower() for k in TPN_KEYWORDS) for l in network_labels])
dmn_mask = np.array([any(k.lower() in l.lower() for k in DMN_KEYWORDS) for l in network_labels])
print(f"TPN parcels: {tpn_mask.sum()}  |  DMN parcels: {dmn_mask.sum()}")

n_gsbs_states = state_patterns.shape[0]
state_tpn_score = np.zeros(n_gsbs_states)
for s in range(n_gsbs_states):
    sp = state_patterns[s]
    tpn_val = sp[tpn_mask].mean() if tpn_mask.sum() > 0 else 0.0
    dmn_val = sp[dmn_mask].mean() if dmn_mask.sum() > 0 else 0.0
    state_tpn_score[s] = tpn_val - dmn_val

print(f"TPN scores: {state_tpn_score.min():.3f} to {state_tpn_score.max():.3f}  mean={state_tpn_score.mean():.3f}")

event_tpn = np.zeros(n_events)
for i, (onset, dur) in enumerate(zip(event_onsets, event_durations)):
    event_states = gsbs_states[int(onset):int(onset + dur)]
    if len(event_states) > 0:
        scores = [state_tpn_score[s] for s in event_states if s < n_gsbs_states]
        event_tpn[i] = float(sum(scores)) / len(scores) if scores else 0.0

print(f"Event TPN scores: mean={event_tpn.mean():.3f}  SD={event_tpn.std():.3f}")


In [ ]:
if n_total > 0:
    episode_tpn   = []
    episode_delta = []
    for ep in model_log:
        ti = ep['target_idx']
        if ti < n_events:
            episode_tpn.append(event_tpn[ti])
            episode_delta.append(ep['delta_sim'])

    episode_tpn   = np.array(episode_tpn)
    episode_delta = np.array(episode_delta)

    r_h4, p_h4 = spearmanr(episode_tpn, episode_delta)
    print(f"H4: TPN score x delta_sim  r = {r_h4:.3f}, p = {p_h4:.4f}")

    med = float(np.median(episode_tpn))
    tpn_high_delta = episode_delta[episode_tpn >= med]
    tpn_low_delta  = episode_delta[episode_tpn <  med]
    U_h4, p_mw = stats.mannwhitneyu(tpn_high_delta, tpn_low_delta, alternative='less')
    print(f"H4 Mann-Whitney (TPN-high delta < TPN-low delta): U={U_h4:.1f}, p={p_mw:.4f}")
    print(f"  TPN-high mean delta = {tpn_high_delta.mean():+.4f}")
    print(f"  TPN-low  mean delta = {tpn_low_delta.mean():+.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ax = axes[0]
    ax.scatter(episode_tpn, episode_delta, alpha=0.3, s=15, color='steelblue')
    if len(episode_tpn) > 2:
        z = np.polyfit(episode_tpn, episode_delta, 1)
        xs = np.linspace(float(episode_tpn.min()), float(episode_tpn.max()), 100)
        ax.plot(xs, np.polyval(z, xs), 'r-', lw=2, label=f'r={r_h4:.3f}, p={p_h4:.3f}')
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel("TPN dominance score (GSBS state projection)")
    ax.set_ylabel("delta cosine similarity")
    ax.set_title("H4: Global brain state x representational change")
    ax.legend()
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    ax = axes[1]
    ax.violinplot([tpn_low_delta, tpn_high_delta], positions=[0, 1], showmedians=True)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['TPN-low (DMN-dominant)', 'TPN-high (TPN-dominant)'])
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_ylabel("delta cosine similarity")
    ax.set_title(f"H4 median split  (p={p_mw:.4f})")
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    plt.suptitle(f"H4: Global brain state modulation  |  {SUB} run {RUN}", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/H4_global_state_{FEATURE_MODALITY}_{SUB}_run{RUN}.png", dpi=150, bbox_inches='tight')
    plt.show()

    print("=" * 60)
    print("H4 RESULT SUMMARY")
    print("=" * 60)
    print(f"  TPN score x delta_sim  r = {r_h4:.3f}, p = {p_h4:.4f}")
    if p_h4 < 0.05 and r_h4 < 0:
        print("  H4 SUPPORTED: higher TPN dominance -> more differentiation (negative delta_sim).")
    elif p_h4 < 0.05 and r_h4 > 0:
        print("  Significant but opposite: TPN-dominant events -> more integration.")
        print("  Consider: model osc_amp may put TPN events in the integration zone.")
    else:
        print("  No significant TPN-delta relationship. Possible reasons:")
        print("  1. TPN/DMN scores derived from same signal as GSBS (circular).")
        print("  2. Insufficient episode variance across states.")
        print("  3. For independent test: use FlippingPaper TPN timeseries as global_state_run{N}.npy")
    print("  Mann-Whitney TPN-high vs TPN-low delta: p =", round(float(p_mw), 4))
    print("=" * 60)
else:
    print("No competition episodes -- H4 cannot be tested.")


**Interpreting H4:** Negative Spearman r (TPN score x delta_sim) supports H4: events during TPN-dominant states tend toward differentiation. The TPN score here is derived from GSBS state patterns projected onto Yeo 17 network labels, which is a reasonable but not fully independent measure. For an independent H4 test, supply a global state timeseries from the FlippingPaper DMN/TPN analysis.

## 9. H1: Hippocampal RSA for differentiated vs integrated pairs

**H1 (single-run adaptation):** Do model-differentiated event pairs show lower hippocampal RSA
than model-integrated pairs, within the current run?

This tests whether the model's competition predictions align with the brain's *existing*
representational geometry in hippocampus-adjacent regions. A significant result (diff < intg)
is consistent with H1 and suggests the NMPH model is capturing real structure in the feature space.


In [ ]:
# Hippocampus-adjacent parcels: Default / Para / Limbic (Yeo 17)
hipp_mask = np.array([
    any(k.lower() in l.lower() for k in ['default', 'para', 'limbic'])
    for l in network_labels
])
hipp_idx = np.where(hipp_mask)[0]
print(f"Hippocampus-adjacent parcels (Default/Para/Limbic): {len(hipp_idx)}")

ev_bold_hipp = ev_bold[:, hipp_idx]
hipp_rsa = np.corrcoef(ev_bold_hipp)   # (n_events, n_events)
print(f"Hippocampal RSA matrix: {hipp_rsa.shape}")

pairs_idx         = np.triu_indices(n_events, k=1)
model_change_flat = rsa_change[pairs_idx]
hipp_rsa_flat     = hipp_rsa[pairs_idx]

is_diff = model_change_flat < -0.01
is_intg = model_change_flat >  0.01
print(f"Model-differentiated pairs: {is_diff.sum()}")
print(f"Model-integrated pairs    : {is_intg.sum()}")


In [ ]:
if is_diff.sum() > 5 and is_intg.sum() > 5:
    U_h1, p_h1 = stats.mannwhitneyu(
        hipp_rsa_flat[is_diff], hipp_rsa_flat[is_intg], alternative='less')
    diff_mean = float(hipp_rsa_flat[is_diff].mean())
    intg_mean = float(hipp_rsa_flat[is_intg].mean())

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    ax.violinplot([hipp_rsa_flat[is_diff], hipp_rsa_flat[is_intg]],
                  positions=[0, 1], showmedians=True)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Model: Differentiation', 'Model: Integration'])
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_ylabel("Hippocampal RSA (Pearson r, current run)")
    ax.set_title(f"H1: Hippocampal RSA by NMPH outcome\n(p = {p_h1:.4f})")
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    ax = axes[1]
    ax.scatter(model_change_flat, hipp_rsa_flat, alpha=0.2, s=12, color='steelblue')
    if len(model_change_flat) > 2:
        z = np.polyfit(model_change_flat, hipp_rsa_flat, 1)
        xs = np.linspace(float(model_change_flat.min()), float(model_change_flat.max()), 200)
        ax.plot(xs, np.polyval(z, xs), 'r-', lw=2)
        r_sc, p_sc = spearmanr(model_change_flat, hipp_rsa_flat)
        ax.set_title(f"Model RSA change x hippocampal RSA\nr = {r_sc:.3f}, p = {p_sc:.4f}")
    ax.axvline(0, color='k', lw=0.8, ls='--'); ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel("Model RSA change (delta_sim)")
    ax.set_ylabel("Hippocampal RSA (current run)")
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    plt.suptitle(f"H1: Hippocampal RSA | {SUB} run {RUN}", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/H1_hippocampal_rsa_{FEATURE_MODALITY}_{SUB}_run{RUN}.png",
                dpi=150, bbox_inches='tight')
    plt.show()

    print("=" * 65)
    print("H1: HIPPOCAMPAL RSA -- DIFFERENTIATED vs INTEGRATED PAIRS")
    print("=" * 65)
    print(f"  Model-differentiated mean hippocampal r : {diff_mean:+.4f}")
    print(f"  Model-integrated mean hippocampal r     : {intg_mean:+.4f}")
    print(f"  Difference                              : {diff_mean - intg_mean:+.4f}")
    print(f"  Mann-Whitney p (one-tailed, diff < intg): {p_h1:.4f}")
    print()
    if p_h1 < 0.05 and diff_mean < intg_mean:
        print("  H1 SUPPORTED (within-run): differentiated pairs show lower hippocampal RSA.")
        print("  The NMPH model identifies pairs that the brain already represents distinctly.")
    elif p_h1 < 0.05 and diff_mean > intg_mean:
        print("  Significant but opposite direction. Check: HRF delay, parcel set, osc_amp.")
    elif diff_mean < intg_mean:
        print("  Trend in predicted direction but not significant.")
        print("  Limited power (1 subject, 1 run). Consider cross-subject averaging.")
    else:
        print("  No effect. Model competition structure may not align with hippocampal RSA")
        print("  for this feature space. Try lower COMPETITOR_THRESH or VGG-19 modality.")
    print()
    print("  Note: 'hippocampal' = Default/Para/Limbic Schaefer parcels (proxy).")
    print("  For a strict test, use a dedicated subcortical hippocampal mask.")
    print("=" * 65)
else:
    print(f"Not enough pairs (diff={is_diff.sum()}, intg={is_intg.sum()}).")
    print("Lower COMPETITOR_THRESH in notebook 02 to generate more competition episodes.")


**Interpreting H1:** H1 is the project's central empirical prediction. A within-run result (diff < intg in hippocampal RSA) is consistent with but weaker than the full longitudinal test (cross-run RSA *change* for matched events across all 8 runs). The longitudinal test requires loading GSBS objects for all 8 runs and matching events across runs by semantic content — a next step once the single-run pipeline is validated.

## 10. Save summary statistics

In [ ]:
summary = dict(
    sub=SUB, run=RUN, modality=FEATURE_MODALITY,
    r_map=r_map,
    p_map=p_map,
    p_map_fdr=p_map_fdr,
    sig_parcels=sig_parcels.tolist(),
    network_r=network_r,
    n_events=n_events,
    n_episodes=n_total,
    n_diff=n_diff,
    n_intg=n_intg,
)

save_path = f"{RESULTS_DIR}/brain_comparison_summary_{FEATURE_MODALITY}_{SUB}_run{RUN}.pkl"
with open(save_path, 'wb') as f:
    pickle.dump(summary, f)
np.save(f"{RESULTS_DIR}/r_map_parcel_{FEATURE_MODALITY}_{SUB}_run{RUN}.npy", r_map)
np.save(f"{RESULTS_DIR}/p_map_fdr_{FEATURE_MODALITY}_{SUB}_run{RUN}.npy", p_map_fdr)
print(f"Saved -> {save_path}")


## 11. GitHub Sync — Push

In [ ]:
import json as _j, subprocess as _sp
from datetime import datetime as _dt
from google.colab import _message

_nb   = _message.blocking_request('get_ipynb', timeout_sec=30)
_dest = f"{REPO_PATH}/{NOTEBOOK_REL}"
import os as _os; _os.makedirs(_os.path.dirname(_dest), exist_ok=True)
with open(_dest, 'w') as _f: _j.dump(_nb, _f, indent=1)
_msg = f"[colab] 03_brain_comparison ({FEATURE_MODALITY}, {SUB}, run{RUN}): {_dt.now():%Y-%m-%d %H:%M}"
_sp.run(["git", "-C", REPO_PATH, "add", NOTEBOOK_REL], check=True)
_res = _sp.run(["git", "-C", REPO_PATH, "commit", "-m", _msg], capture_output=True, text=True)
if "nothing to commit" in _res.stdout:
    print("Nothing to commit.")
else:
    _sp.run(["git", "-C", REPO_PATH, "push"], check=True)
    print(f"Pushed: {_msg}")
